# Ingestion Layer (CSV / KML → Bronze Delta)

Reads every source file from the Unity Catalog Volume and lands it into a **bronze Delta table** using Databricks Auto Loader (`cloudFiles`). Each run only processes **new files** thanks to the per-table checkpoint; re-running is a safe no-op for the CSV streams.

| Source | Volume path | Bronze table |
|---|---|---|
| ADI claimant | `landing/**/ADI_claimant*.csv` | `bronze_adi_claimant` |
| ADI crime | `landing/**/ADI_crime*.csv` | `bronze_adi_crime` |
| ADI health | `landing/**/ADI_health*.csv` | `bronze_adi_health` |
| Price-paid | `landing/price_paid/pp-*.csv` | `bronze_price_paid` |
| Postcode (ONSPD) | `landing/postcode/ONSPD_*.csv` | `bronze_postcode_lookup` |
| Police crime (per force) | `landing/**/*<force>-street*.csv` | `bronze_<force>_crime` |
| Police boundary (per force) | `landing/**/*<force>.kml` | `bronze_<force>_boundary` |

Police forces covered (driven by `POLICE_FORCES` in section 1, easy to extend):
Cambridgeshire, Avon and Somerset, Merseyside, Nottinghamshire.

Every table carries two provenance columns:
- `_source_file` — full Volume path of the originating file
- `_ingest_ts` — UTC timestamp when the row was landed

**Run order:** this notebook must run after `00_setup.ipynb` and before `cleaning_and_validation_layer.ipynb`.

## 1. Imports & Configuration

In [ ]:
import glob
from datetime import datetime

import geopandas as gpd
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType
)

# ── Catalog / schema ───────────────────────────────────────────────
CATALOG = "crime_data"   # change if your workspace catalog differs
SCHEMA  = "bronze"

# ── Volume paths ───────────────────────────────────────────────────
# Schemas + volumes are provisioned by notebooks/00_setup.ipynb — run that first.
# CSVs are uploaded to `landing`, preserving the original folder structure.
LANDING = f"/Volumes/{CATALOG}/{SCHEMA}/landing"

# Auto Loader writes checkpoints and inferred-schema logs here.
META = f"/Volumes/{CATALOG}/{SCHEMA}/_meta"

# ── Police forces (scalable: append a force here to onboard it) ──────────────────
# Names follow the UK Police Data filename convention (lowercase, hyphenated).
POLICE_FORCES = [
    "cambridgeshire",
    "avon-and-somerset",
    "merseyside",
    "nottinghamshire",
]

print(f"Catalog       : {CATALOG}")
print(f"Schema        : {SCHEMA}")
print(f"Landing       : {LANDING}")
print(f"Meta          : {META}")
print(f"Police forces : {POLICE_FORCES}")

## 2. Auto Loader Helpers

Two helpers wrap the read → (optional transform) → write pattern: one streaming helper for CSVs (used by every CSV source) and one batch helper for KML boundary files (geopandas can't be driven from `cloudFiles`).

In [ ]:
def autoload_to_delta(
    *,
    source_dir,
    glob_pattern,
    table,
    schema=None,
    schema_hints=None,
    csv_options=None,
    transform=None,
):
    """Stream CSVs matching glob_pattern from source_dir into bronze Delta table.

    source_dir    : sub-path under LANDING (e.g. 'price_paid'); '' for landing root
    glob_pattern  : filename glob (e.g. '*merseyside-street*.csv')
    table         : unqualified table name; written to CATALOG.SCHEMA.table
    schema        : explicit StructType — skips inference (use for headerless CSVs)
    schema_hints  : cloudFiles.schemaHints string for partial typing
    csv_options   : dict of extra CSV reader options
    transform     : df -> df applied after provenance columns are attached

    Uses Trigger.AvailableNow so the stream consumes all pending files then stops,
    behaving like a batch job while still tracking which files have been processed.
    """
    full_table = f"{CATALOG}.{SCHEMA}.{table}"
    source_path = f"{LANDING}/{source_dir}"

    reader = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", f"{META}/_schemas/{table}")
        .option("pathGlobFilter", glob_pattern)
        .option("recursiveFileLookup", "true")
        .option("header", "true")
    )

    if schema:
        reader = reader.schema(schema)
    if schema_hints:
        reader = reader.option("cloudFiles.schemaHints", schema_hints)
    if csv_options:
        for k, v in csv_options.items():
            reader = reader.option(k, v)

    df = reader.load(source_path)

    # Provenance columns available on every streaming source via _metadata
    df = (
        df
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_ingest_ts",   F.current_timestamp())
    )

    if transform:
        df = transform(df)

    query = (
        df.writeStream
        .option("checkpointLocation", f"{META}/_checkpoints/{table}")
        .trigger(availableNow=True)
        .toTable(full_table)
    )
    query.awaitTermination()
    print(f"Done: {full_table}")

In [ ]:
def autoload_kml_to_delta(
    *,
    source_dir,
    glob_pattern,
    table,
    transform=None,
):
    """Load KML boundary files into a bronze Delta table.

    Geopandas can't be driven from cloudFiles, so this is a batch append.
    Geometry is serialised to WKT (string) so the table is portable.
    """
    full_table = f"{CATALOG}.{SCHEMA}.{table}"
    source_path = f"{LANDING}/{source_dir}"

    kml_files = glob.glob(f"{source_path}/**/{glob_pattern}", recursive=True)

    if not kml_files:
        print(f"No files matched: {source_path}/**/{glob_pattern}")
        return

    dfs = []
    for filepath in kml_files:
        gdf = gpd.read_file(filepath, driver='KML')
        gdf['geometry_wkt'] = gdf['geometry'].apply(lambda geom: geom.wkt if geom else None)
        gdf = gdf.drop(columns=['geometry'])
        gdf['_source_file'] = filepath
        gdf['_ingest_ts'] = datetime.now()
        dfs.append(gdf)

    combined = pd.concat(dfs, ignore_index=True)
    df = spark.createDataFrame(combined)

    if transform:
        df = transform(df)

    df.write.format("delta").mode("append").saveAsTable(full_table)
    print(f"Done: {full_table} ({len(combined)} rows from {len(kml_files)} file(s))")

## 3. ADI Ingest (Claimant / Crime / Health)

Fifteen CSVs across five year folders. Schema is inferred by Auto Loader (small files, well-structured headers). `year` is derived from the folder name in `_source_file` — the CSV itself has no year column.

Folder layout in the Volume:
```
landing/adi/
    ADI_2017/ADI_claimant_counts_2017.csv
    ADI_2017/ADI_crime_2017.csv
    ADI_2017/ADI_health_2017.csv
    ADI_2018/...
    ...
```

In [ ]:
def add_adi_year(df):
    """Extract year (int) from the ADI_<YYYY> folder name embedded in _source_file."""
    return df.withColumn(
        "year",
        F.regexp_extract(F.col("_source_file"), r"ADI_(\d{4})", 1).cast("int"),
    )

In [ ]:
# ADI claimant
autoload_to_delta(
    source_dir="",
    glob_pattern="*claimant*.csv",
    table="bronze_adi_claimant",
    transform=add_adi_year,
)

In [ ]:
# ADI crime
autoload_to_delta(
    source_dir="",
    glob_pattern="ADI_crime*.csv",
    table="bronze_adi_crime",
    transform=add_adi_year,
)

In [ ]:
# ADI health
autoload_to_delta(
    source_dir="",
    glob_pattern="*health*.csv",
    table="bronze_adi_health",
    transform=add_adi_year,
)

## 4. Price-Paid Ingest

Six CSVs (`pp-2017-part1.csv`, `pp-2017-part2.csv`, `pp-2018.csv`, …). The Land Registry PPD format is **headerless** — the 16-column schema is supplied explicitly so Auto Loader doesn't misread the first row as a header.

All columns land as `StringType`. Type casting (`price → long`, `date_of_transfer → date`) is left to the cleaning layer, matching the existing pipeline design.

In [ ]:
# Standard 16-column Land Registry PPD schema (no header in source files)
PPD_COLUMNS = [
    "transaction_id", "price", "date_of_transfer", "postcode",
    "property_type", "old_new", "duration", "paon", "saon",
    "street", "locality", "town_city", "district", "county",
    "ppd_category_type", "record_status",
]

PPD_SCHEMA = StructType(
    [StructField(col, StringType(), True) for col in PPD_COLUMNS]
)

autoload_to_delta(
    source_dir="price_paid",
    glob_pattern="pp-*.csv",
    table="bronze_price_paid",
    schema=PPD_SCHEMA,
    csv_options={"header": "false"},
)

## 5. Postcode Lookup Ingest (ONSPD)

One 1.3 GB CSV with 51 columns. Only `pcds` (postcode) and `lsoa11` (LSOA code) are needed downstream. Using `schemaHints` types those two columns and the `transform` selects only them — the 49 unused columns are never materialised in Delta.

In [ ]:
def select_postcode_cols(df):
    """Keep only the two columns needed downstream plus the provenance columns."""
    return df.select("pcds", "lsoa11", "_source_file", "_ingest_ts")


autoload_to_delta(
    source_dir="postcode",
    glob_pattern="ONSPD_*.csv",
    table="bronze_postcode_lookup",
    schema_hints="pcds string, lsoa11 string",
    transform=select_postcode_cols,
)

## 6. Crime Data Ingest (per police force)

Standard UK Police Data: monthly street-level CSVs (`YYYY-MM-<force>-street.csv`) plus a boundary KML (`<force>.kml`). One CSV per month per force; one KML per force.

The loop below iterates over `POLICE_FORCES` (defined in section 1). Each force gets two bronze tables: `bronze_<force>_crime` and `bronze_<force>_boundary`. Hyphens in force names (e.g. `avon-and-somerset`) are normalised to underscores for the Delta table identifier.

Expected Volume layout:
```
landing/
    2020-01-merseyside-street.csv
    2020-02-merseyside-street.csv
    ...
    merseyside.kml
    2020-01-nottinghamshire-street.csv
    ...
```

In [ ]:
def rename_columns(df):
    """Snake-case the standard UK Police Data column headers."""
    return df \
        .withColumnRenamed("Crime ID",              "crime_id") \
        .withColumnRenamed("Reported by",           "reported_by") \
        .withColumnRenamed("Falls within",          "falls_within") \
        .withColumnRenamed("LSOA code",             "lsoa_code") \
        .withColumnRenamed("LSOA name",             "lsoa_name") \
        .withColumnRenamed("Crime type",            "crime_type") \
        .withColumnRenamed("Last outcome category", "last_outcome_category")

In [ ]:
for force in POLICE_FORCES:
    table_force = force.replace("-", "_")  # Delta identifiers can't contain hyphens

    autoload_to_delta(
        source_dir="",
        glob_pattern=f"*{force}-street*.csv",
        table=f"bronze_{table_force}_crime",
        transform=rename_columns,
    )

    autoload_kml_to_delta(
        source_dir="",
        glob_pattern=f"*{force}.kml",
        table=f"bronze_{table_force}_boundary",
    )

## 7. Ingest Validation

Per the pipeline brief: log row counts at the ingest boundary and confirm every table has at least one row per expected source file. Covers all five static sources plus both per-force tables for every entry in `POLICE_FORCES`.

In [ ]:
BRONZE_TABLES = [
    "bronze_adi_claimant",
    "bronze_adi_crime",
    "bronze_adi_health",
    "bronze_price_paid",
    "bronze_postcode_lookup",
] + [f"bronze_{f.replace('-', '_')}_crime"    for f in POLICE_FORCES] \
  + [f"bronze_{f.replace('-', '_')}_boundary" for f in POLICE_FORCES]

print("=== Bronze layer — ingest summary ===")
for table in BRONZE_TABLES:
    full = f"{CATALOG}.{SCHEMA}.{table}"
    df   = spark.read.table(full)
    rows = df.count()
    files = df.select("_source_file").distinct().count()
    print(f"\n{table}")
    print(f"  total rows  : {rows:,}")
    print(f"  source files: {files}")
    df.groupBy("_source_file").count().orderBy("_source_file").show(truncate=False)

    # Guard: every row must have provenance
    null_prov = df.filter(F.col("_source_file").isNull() | F.col("_ingest_ts").isNull()).count()
    assert null_prov == 0, f"{table}: {null_prov:,} rows missing provenance columns"

print("\nAll provenance checks passed.")